# 02 · Synthetic Supplier Quotes (calibrated to real data)
Supplier quotations are confidential, so they are **simulated** — but every quote inherits the real macro indices for its month and `unit_price` is an explicit function of them. Produced by `python -m src.data_generation`.

In [ ]:
%matplotlib inline
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40, "display.width", 160)
from src import config as C
print("project root:", ROOT)

In [ ]:
q = pd.read_csv(C.RAW_QUOTES_CSV)
master = pd.read_csv(C.SUPPLIER_MASTER_CSV)
print(q.shape, '| suppliers:', q.supplier_id.nunique(), '| months:', q.groupby(['year','month']).ngroups)
q.head()

## Star schema
A **fact table** of quotes + a **supplier dimension** table (which also holds the latent generating archetype, used later only for validation).

In [ ]:
master.head()

## Do the encoded relationships hold?
These are the assumptions the brief requires; we verify them directly in the generated data.

In [ ]:
# (a) material-intensive part tracks the real commodity index
ac = q[q.component_type=='Aluminum Casting']
print('Aluminum Casting  r(raw_material_index, unit_price) =', round(ac.raw_material_index.corr(ac.unit_price),2))

# (b) Spot is the most volatile contract type (within component)
cv = (q.groupby(['component_type','contract_type']).unit_price.apply(lambda s: s.std()/s.mean())
        .groupby('contract_type').mean().round(3))
print('\nwithin-component price CV by contract type:'); print(cv)

# (c) economies of scale
print('\nr(log order_volume, price/median) =', round(np.log(q.order_volume).corr(q.unit_price/q.groupby('component_type').unit_price.transform('median')),2))

In [ ]:
order = q.groupby('component_type').unit_price.median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(11,6))
sns.boxplot(data=q, y='component_type', x='unit_price', order=order, color='#4c72b0', fliersize=1)
ax.set_xscale('log'); ax.set_title('Unit price by component type (log)'); plt.show()

Battery Module & Electric Motor Part dominate the price scale; commodities/energy/FX drive the variation *within* each component tier.